# World University Rankings 2026 – ETL desde Times Higher Education

Este notebook extrae, transforma y guarda los datos de **tres rankings publicados por _Times Higher Education_ (THE)** 
directamente desde su sitio web oficial:

1. **World University Rankings 2026** (ranking global).
2. **World University Rankings by Subject 2026 – Business & Economics** (ranking por disciplina).
3. **Latin America University Rankings 2026** (ranking regional para Latinoamérica).

El enfoque se mantiene fiel a la idea original (_web scraping_ con Selenium + BeautifulSoup) 
pero se adapta a la nueva estructura del sitio de THE, se usa `webdriver-manager` para evitar descargar 
manualmente `chromedriver.exe`, y se soporta la nueva metodología de puntajes (Teaching, Research Environment, 
Research Quality, Industry e International Outlook).

> Origen de las URLs: archivo **`Web Scrapping from Times higher Education_Latest.txt`**.

## 1. Objetivo

Automatizar la captura de la información publicada por THE y dejarla lista para análisis posterior 
(limpieza, visualización, modelado). El _web scraping_ se usa porque THE no expone un API público estable; 
por eso se recorre la tabla HTML que el sitio construye dinámicamente con JavaScript.


## 2. Prerrequisitos

* Python 3.9 o superior.
* Jupyter Notebook o cualquier entorno que permita ejecutar Python.
* Librerías: `beautifulsoup4`, `selenium`, `webdriver-manager`, `requests`, `pandas`, `lxml`.
* Navegador **Google Chrome** o **Chromium** instalado. `webdriver-manager` se encarga de descargar 
el driver compatible automáticamente (ya **no** hace falta un `chromedriver.exe` manual).

Instalar dependencias:
```bash
pip install -r requirements.txt
```


## 3. ¿Qué datos vamos a extraer?

Antes de escribir código, conviene entender la página. THE presenta los rankings en una **tabla con columnas** 
para el puesto, nombre de la universidad, puntajes y pilares metodológicos. La vista que el usuario observa 
en el navegador se parece a esto:

![Vista del ranking THE 2026](img/basic_page_01.PNG)
<br/><br/>


Sin embargo, para extraer los datos de forma confiable no miramos la pantalla sino el **HTML renderizado**. 
Presionando `F12` en Chrome abrimos las _Developer Tools_ y podemos ubicar los selectores CSS que usaremos 
en el scraping:

![Inspección del HTML en DevTools](img/page_code_02.PNG)
<br/><br/>

Las columnas de la nueva metodología (2026) son: **Overall, Teaching, Research Environment, Research Quality, 
Industry e International Outlook**. Además capturaremos _Number of FTE students_, _Student/Staff ratio_, 
_% International students_, _Female/Male ratio_, país y dirección completa de cada universidad.

## 4. Implementación en Python

El sitio de THE carga la tabla de rankings mediante **AJAX/JavaScript**, por lo que una simple petición 
con `requests` devolvería la plantilla vacía. Por eso usamos **Selenium**, que controla un navegador real 
y nos entrega el HTML después de que el JS termina de ejecutarse. Una vez tenemos el HTML listo, 
**BeautifulSoup** se encarga de localizar los selectores y extraer el texto de cada celda.

In [10]:
# Librerías estándar
import json
import re
import time
from urllib.parse import urljoin

# Librerías de terceros
import pandas as pd
import requests
from bs4 import BeautifulSoup as soup

# Selenium + webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


### 4.1 Definir las URLs de los tres rankings

Las URLs fueron obtenidas del archivo `Web Scrapping from Times higher Education_Latest.txt`. 
Agregamos un diccionario que mapea cada nombre lógico al link y a su metodología para dejar trazabilidad.

In [11]:
RANKINGS = {
    'world_2026': {
        'nombre': 'World University Rankings 2026',
        'url':    'https://www.timeshighereducation.com/world-university-rankings/latest/world-ranking',
        'metodologia': 'https://www.timeshighereducation.com/world-university-rankings/methodology',
        'archivo_salida': 'the_world_ranking_2026.csv',
    },
    'business_economics_2026': {
        'nombre': 'Subject Ranking 2026 – Business & Economics',
        'url':    'https://www.timeshighereducation.com/world-university-rankings/2026/subject-ranking/business-and-economics',
        'metodologia': 'https://www.timeshighereducation.com/world-university-rankings/world-university-rankings-subject-2026-methodology',
        'archivo_salida': 'the_subject_business_economics_2026.csv',
    },
    'latam_2026': {
        'nombre': 'Latin America University Rankings 2026',
        'url':    'https://www.timeshighereducation.com/world-university-rankings/2026/latin-america-university-rankings',
        'metodologia': 'https://www.timeshighereducation.com/world-university-rankings/latin-america-university-rankings-2026-methodology',
        'archivo_salida': 'the_latam_ranking_2026.csv',
    },
}

for k, v in RANKINGS.items():
    print(f"{k:25s} -> {v['nombre']}")


world_2026                -> World University Rankings 2026
business_economics_2026   -> Subject Ranking 2026 – Business & Economics
latam_2026                -> Latin America University Rankings 2026


### 4.2 Inicializar el driver de Chrome

`webdriver-manager` descarga automáticamente la versión de `chromedriver` que corresponde al Chrome instalado. 
Usamos modo `headless` (sin ventana visible) para que el scraping sea más liviano.

In [12]:
def crear_driver(headless: bool = True) -> webdriver.Chrome:
    """Crea un driver de Chrome con las opciones recomendadas para scraping."""
    options = Options()
    if headless:
        options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_experimental_option('excludeSwitches', ['enable-automation'])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    # Ocultar navigator.webdriver para evitar detección
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
    })
    return driver

### 4.3 Función robusta de scraping

THE ha ido cambiando los nombres de las clases CSS entre versiones. Para evitar depender de una única 
estructura, la función `scrapear_ranking` aplica una estrategia en cascada:

1. Carga la página con Selenium y espera a que aparezca la tabla.
2. Busca un `<script>` con JSON embebido (patrón usado por THE: atributo `data-rankings` o `application/json`).
3. Si no encuentra JSON, recorre la **tabla HTML** usando múltiples selectores alternativos.
4. Devuelve un `DataFrame` homogéneo con las columnas comunes a los tres rankings.

In [13]:
# Se usan patrones regex para mayor robustez ante cambios menores de versión.
COLUMNAS_CSS = {
    'rank':                       ['rank-position', 'rank'],
    'university':                 ['ranking-institution-title', 'name'],
    'country':                    ['ranking-institution-country', 'country-name', 'country'],
    'overall_score':              ['overall-score', 'scores-score'],
    'teaching_score':             ['teaching-score', 'teaching'],
    'research_env_score':         ['research-env-score', 'research-environment'],
    'research_quality_score':     ['research-quality-score', 'research-quality'],
    'industry_score':             ['industry-score', 'industry'],
    'international_outlook_score':['international-outlook-score', 'international-outlook'],
}


def _celda(fila, alias_clases: list) -> str | None:
    """Busca la primera <td> que contenga alguna de las clases indicadas."""
    for cls in alias_clases:
        td = fila.find('td', class_=re.compile(re.escape(cls), re.I))
        if td:
            return td.get_text(strip=True)
    return None


In [14]:
# ── Función principal de scraping (CORREGIDA) ──────────────────────────────────
def scrapear_ranking(driver: webdriver.Chrome, url: str, espera_seg: int = 30) -> pd.DataFrame:
    """Descarga la tabla de un ranking de THE y devuelve un DataFrame.

    Estrategia 1: extrae JSON embebido en <script> (window.__INITIAL_STATE__ o similar).
    Estrategia 2: parsea directamente el <table> del HTML renderizado.
    """
    # CORRECCIÓN: se eliminó el sufijo #!/length/-1 que ya no funciona en la versión actual de THE
    print(f'→ Cargando {url}')
    driver.get(url)

    # Esperar a que aparezca al menos una fila de la tabla
    try:
        WebDriverWait(driver, espera_seg).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'table tbody tr'))
        )
    except Exception as e:
        print('  · Advertencia: no se detectó <tr> visible, seguimos de todos modos ->', e)

    # Scroll progresivo para forzar la carga lazy de todas las filas
    altura_total = driver.execute_script('return document.body.scrollHeight')
    paso = 800
    for pos in range(0, altura_total, paso):
        driver.execute_script(f'window.scrollTo(0, {pos});')
        time.sleep(0.3)
    driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
    time.sleep(3)
    driver.execute_script('window.scrollTo(0, 0);')
    time.sleep(1)

    html = driver.page_source
    pagina = soup(html, 'html.parser')

    # ── Estrategia 1: JSON embebido en <script> ────────────────────────────────
    # CORRECCIÓN: se amplían los patrones de búsqueda y se corrige la precedencia
    # del operador `or/and` que causaba falsos positivos en el código original.
    for script in pagina.find_all('script'):
        raw = script.string or ''

        # Patrón A: window.__INITIAL_STATE__ o similar con datos de rankings
        if ('rankings_data' in raw or 'rankedInstitutions' in raw
                or ('\"rank\"' in raw and '\"name\"' in raw and '\"scores\"' in raw)):
            try:
                # Buscar array JSON que contenga objetos con rank y name
                m = re.search(
                    r'(\[\s*\{\s*"(?:rank|name|institution)"[\s\S]+?\}\s*\])',
                    raw
                )
                if m:
                    datos = json.loads(m.group(1))
                    df = pd.DataFrame(datos)
                    if not df.empty and any(c in df.columns for c in ('name', 'institution', 'rank')):
                        print(f'  · Datos obtenidos vía JSON embebido (patrón A): {len(df)} filas')
                        return df
            except Exception:
                pass

        # Patrón B: JSON-LD con ItemList (schema.org)
        if script.get('type') == 'application/ld+json':
            try:
                datos_ld = json.loads(raw)
                items = datos_ld.get('itemListElement', [])
                if items:
                    registros = []
                    for item in items:
                        inst = item.get('item', {})
                        registros.append({
                            'rank':          item.get('position'),
                            'name':          inst.get('name'),
                            'url':           inst.get('url'),
                            'country':       inst.get('address', {}).get('addressCountry'),
                            'overall_score': inst.get('aggregateRating', {}).get('ratingValue'),
                        })
                    df = pd.DataFrame(registros)
                    if not df.empty:
                        print(f'  · Datos obtenidos vía JSON-LD (patrón B): {len(df)} filas')
                        return df
            except Exception:
                pass

    # ── Estrategia 2: parseo directo del <table> ───────────────────────────────
    # CORRECCIÓN: se usan las clases CSS reales de THE y se añade fallback por índice de columna.
    filas = pagina.select('table tbody tr')
    print(f'  · Filas detectadas en la tabla: {len(filas)}')

    registros = []
    for tr in filas:
        # Intentar obtener el nombre por clase CSS real
        td_name = (
            tr.find('td', class_=re.compile(r'ranking-institution-title', re.I))
            or tr.find('td', class_=re.compile(r'name', re.I))
        )

        # Fallback: segunda celda si no se encontró por clase
        if not td_name:
            tds = tr.find_all('td')
            if len(tds) < 2:
                continue
            td_name = tds[1]

        a = td_name.find('a')
        nombre = a.get_text(strip=True) if a else td_name.get_text(strip=True)
        href   = a.get('href') if a else None

        registro = {
            'rank':                        _celda(tr, COLUMNAS_CSS['rank']),
            'name':                        nombre,
            'url':                         urljoin('https://www.timeshighereducation.com', href) if href else None,
            'country':                     _celda(tr, COLUMNAS_CSS['country']),
            'overall_score':               _celda(tr, COLUMNAS_CSS['overall_score']),
            'teaching_score':              _celda(tr, COLUMNAS_CSS['teaching_score']),
            'research_env_score':          _celda(tr, COLUMNAS_CSS['research_env_score']),
            'research_quality_score':      _celda(tr, COLUMNAS_CSS['research_quality_score']),
            'industry_score':              _celda(tr, COLUMNAS_CSS['industry_score']),
            'international_outlook_score': _celda(tr, COLUMNAS_CSS['international_outlook_score']),
        }
        registros.append(registro)

    df = pd.DataFrame(registros)
    if df.empty:
        print('  · ¡Atención! No se extrajeron filas. '
              'Considera ejecutar con headless=False para inspeccionar el HTML manualmente.')
    else:
        # Eliminar filas completamente vacías (filas de agrupación/separador)
        df.dropna(how='all', inplace=True)
        df.reset_index(drop=True, inplace=True)
    return df

### 4.4 Ejecutar el scraping de los tres rankings

Abrimos un único driver y reutilizamos la sesión para los tres rankings. Los resultados se guardan en 
un diccionario para procesarlos más abajo.

In [15]:
# ── Ejecutar el scraping de los tres rankings ──────────────────────────────────
driver = crear_driver(headless=True)
resultados = {}
try:
    for clave, cfg in RANKINGS.items():
        print(f'\n=== {cfg["nombre"]} ===')
        resultados[clave] = scrapear_ranking(driver, cfg['url'])
        print(f'  · Filas recolectadas: {len(resultados[clave])}')
        # Guardar CSV
        if not resultados[clave].empty:
            resultados[clave].to_csv(cfg['archivo_salida'], index=False, encoding='utf-8-sig')
            print(f'  · Guardado en: {cfg["archivo_salida"]}')
finally:
    driver.quit()

# Vista rápida
for k, df in resultados.items():
    print(f'\n>>> {k} ({len(df)} filas)')
    display(df.head())


=== World University Rankings 2026 ===
→ Cargando https://www.timeshighereducation.com/world-university-rankings/latest/world-ranking
  · Filas detectadas en la tabla: 35
  · Filas recolectadas: 29
  · Guardado en: the_world_ranking_2026.csv

=== Subject Ranking 2026 – Business & Economics ===
→ Cargando https://www.timeshighereducation.com/world-university-rankings/2026/subject-ranking/business-and-economics
  · Filas detectadas en la tabla: 35
  · Filas recolectadas: 29
  · Guardado en: the_subject_business_economics_2026.csv

=== Latin America University Rankings 2026 ===
→ Cargando https://www.timeshighereducation.com/world-university-rankings/2026/latin-america-university-rankings
  · Filas detectadas en la tabla: 34
  · Filas recolectadas: 28
  · Guardado en: the_latam_ranking_2026.csv

>>> world_2026 (29 filas)


,rank,name,url,country,overall_score,teaching_score,research_env_score,research_quality_score,industry_score,international_outlook_score
0,None,University of Oxford,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
1,None,Massachusetts Institute of Technology,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
2,None,Princeton University,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
3,None,University of Cambridge,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
4,None,Harvard University,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None



>>> business_economics_2026 (29 filas)


,rank,name,url,country,overall_score,teaching_score,research_env_score,research_quality_score,industry_score,international_outlook_score
0,None,Massachusetts Institute of Technology,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
1,None,Stanford University,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
2,None,Tsinghua University,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
3,None,University of Oxford,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
4,None,"University of California, Berkeley",https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None



>>> latam_2026 (28 filas)


,rank,name,url,country,overall_score,teaching_score,research_env_score,research_quality_score,industry_score,international_outlook_score
0,None,University of São Paulo,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
1,None,University of Campinas,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
2,None,Pontificia Universidad Católica de Chile,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
3,None,Federal University of Rio de Janeiro,https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None
4,None,Universidade Estadual Paulista (Unesp),https://www.timeshighereducation.com/world-uni...,None,None,None,None,None,None,None


### 4.5 (Opcional) Enriquecer con la dirección de cada universidad

Igual que en la versión original, recorremos la página de perfil de cada universidad para leer su 
dirección desde el JSON-LD (`<script type="application/ld+json">`). Esta parte **tarda** porque hay una 
petición por universidad; puedes saltártela poniendo `OBTENER_DIRECCIONES = False`.

In [ ]:
OBTENER_DIRECCIONES = False  # <- cambia a False para omitir esta sección

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                        'AppleWebKit/537.36 (KHTML, like Gecko) '
                        'Chrome/124.0.0.0 Safari/537.36'}

def _buscar_en_json(obj, clave):
    """Búsqueda recursiva sencilla, reemplaza a objectpath (descontinuado)."""
    if isinstance(obj, dict):
        if clave in obj:
            return obj[clave]
        for v in obj.values():
            r = _buscar_en_json(v, clave)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = _buscar_en_json(v, clave)
            if r is not None:
                return r
    return None


def obtener_direccion(url_perfil: str, timeout: int = 15) -> dict:
    """Extrae la dirección desde la página de perfil de la universidad."""
    vacio = {
        'full_address': None, 'street_address': None, 'locality': None,
        'region': None, 'postal_code': None, 'country_address': None,
    }
    if not url_perfil:
        return vacio
    try:
        r = requests.get(url_perfil, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        pagina = soup(r.text, 'html.parser')
        # JSON-LD
        jsonld = None
        for tag in pagina.find_all('script', {'type': 'application/ld+json'}):
            try:
                jsonld = json.loads(tag.string or '{}')
                if _buscar_en_json(jsonld, 'address') is not None:
                    break
            except Exception:
                jsonld = None
        direccion = _buscar_en_json(jsonld or {}, 'address') or {}
        full = None
        bloque = pagina.find('div', {'class': re.compile('institution-info__contact-detail.*address')})
        if bloque:
            full = bloque.get_text(strip=True)
        return {
            'full_address':   full,
            'street_address': direccion.get('streetAddress') if isinstance(direccion, dict) else None,
            'locality':       direccion.get('addressLocality') if isinstance(direccion, dict) else None,
            'region':         direccion.get('addressRegion') if isinstance(direccion, dict) else None,
            'postal_code':    direccion.get('postalCode') if isinstance(direccion, dict) else None,
            'country_address':direccion.get('addressCountry') if isinstance(direccion, dict) else None,
        }
    except Exception as e:
        print(f'  ! {url_perfil} -> {e}')
        return vacio


if OBTENER_DIRECCIONES:
    for clave, df in resultados.items():
        if df.empty or 'url' not in df.columns:
            continue
        print(f'\n>>> Enriqueciendo direcciones para {clave} ({len(df)} universidades)')
        direcciones = []
        for i, u in enumerate(df['url'].tolist(), 1):
            direcciones.append(obtener_direccion(u))
            if i % 25 == 0:
                print(f'   {i}/{len(df)} listos')
        df_addr = pd.DataFrame(direcciones)
        resultados[clave] = pd.concat([df.reset_index(drop=True), df_addr.reset_index(drop=True)], axis=1)



>>> Enriqueciendo direcciones para world_2026 (28 universidades)
   25/28 listos

>>> Enriqueciendo direcciones para business_economics_2026 (28 universidades)
   25/28 listos

>>> Enriqueciendo direcciones para latam_2026 (28 universidades)
   25/28 listos


### 4.6 Limpieza y normalización

Homogeneizamos tipos, removemos símbolos (`%`, `,`) y normalizamos el `rank` eliminando sufijos 
como `=`, `+` o rangos (`201–250` -> `201`).

In [8]:
def limpiar(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    if 'intl_students' in df.columns:
        df['intl_students'] = df['intl_students'].astype(str).str.replace('%', '', regex=False)
    if 'rank' in df.columns:
        df['rank_raw'] = df['rank']
        df['rank'] = (df['rank'].astype(str)
                                 .str.replace(r'[=+]', '', regex=True)
                                 .str.replace(r'[–-]\d+', '', regex=True)
                                 .str.strip())
    if 'overall_score' in df.columns:
        df['overall_score'] = df['overall_score'].astype(str).str.replace(r'.*[–-]', '', regex=True)
    if 'number_students' in df.columns:
        df['number_students'] = df['number_students'].astype(str).str.replace(',', '', regex=False)
    df = df.replace({'n/a': None, 'N/A': None, '': None})
    return df


resultados = {k: limpiar(v) for k, v in resultados.items()}
for k, df in resultados.items():
    print(f'{k}: {df.shape}')
    display(df.head())

world_2026: (28, 21)


,rank,name,url,country,overall_score,teaching_score,research_env_score,research_quality_score,industry_score,international_score,...,student_staff_ratio,intl_students,female_male_ratio,full_address,street_address,locality,region,postal_code,country_address,rank_raw
0,NaN,University of Oxford,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,Wellington Square University Offices,Oxford,Oxfordshire,OX1 2JD,United Kingdom,None
1,NaN,Massachusetts Institute of Technology,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,77 Massachusetts Avenue,Cambridge,Massachusetts,02139-4307,United States,None
2,NaN,Princeton University,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,Princeton,Princeton,New Jersey,08544,United States,None
3,NaN,University of Cambridge,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,"[The Old Schools, Trinity Lane]",Cambridge,Cambridgeshire,CB2 1TN,United Kingdom,None
4,NaN,Harvard University,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,Massachusetts Hall,Cambridge,Massachusetts,02138,United States,None


business_economics_2026: (28, 21)


,rank,name,url,country,overall_score,teaching_score,research_env_score,research_quality_score,industry_score,international_score,...,student_staff_ratio,intl_students,female_male_ratio,full_address,street_address,locality,region,postal_code,country_address,rank_raw
0,NaN,Massachusetts Institute of Technology,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,77 Massachusetts Avenue,Cambridge,Massachusetts,02139-4307,United States,None
1,NaN,Stanford University,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,450 Jane Stanford Way,Stanford,California,94305–2004,United States,None
2,NaN,Tsinghua University,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,1 Qinqhuayuan,Beijing,Hebei Province,100084,China,None
3,NaN,University of Oxford,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,Wellington Square University Offices,Oxford,Oxfordshire,OX1 2JD,United Kingdom,None
4,NaN,"University of California, Berkeley",https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,110 Sproul Hall,Berkeley,California,94704,United States,None


latam_2026: (28, 21)


,rank,name,url,country,overall_score,teaching_score,research_env_score,research_quality_score,industry_score,international_score,...,student_staff_ratio,intl_students,female_male_ratio,full_address,street_address,locality,region,postal_code,country_address,rank_raw
0,NaN,University of São Paulo,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,"[374, Rua da Reitoria Cidade Universitaria]",Sao Paulo,Sao Paulo,05508-220,Brazil,None
1,NaN,University of Campinas,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,"[Rua da Reitoria, 121 Barao Geraldo Campinas]",Sao Paulo,Sao Paulo,13083-872,Brazil,None
2,NaN,Pontificia Universidad Católica de Chile,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,Avenida Libertador Bernardo O Higgins 340,Santiago,Santiago,8331150,Chile,None
3,NaN,Federal University of Rio de Janeiro,https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,Rua Antônio Barros de Castro 119 Cidade Univer...,NaN,CEP,21941-853,Brazil,None
4,NaN,Universidade Estadual Paulista (Unesp),https://www.timeshighereducation.com/world-uni...,None,NaN,None,None,None,None,None,...,None,NaN,None,None,Rua Quirino de Andrade 215,Sao Paulo,Sao Paulo,01049-010,Brazil,None


### 4.7 Guardar los resultados

Guardamos un `.csv` por ranking (UTF-8 con BOM para que abra limpio en Excel) en la misma carpeta del notebook.

In [9]:
for clave, cfg in RANKINGS.items():
    df = resultados.get(clave)
    if df is None or df.empty:
        print(f'! {clave} sin datos, se omite')
        continue
    salida = cfg['archivo_salida']
    df.to_csv(salida, index=False, encoding='utf-8-sig')
    print(f'✔ {clave:25s} -> {salida} ({len(df)} filas)')

✔ world_2026                -> the_world_ranking_2026.csv (28 filas)
✔ business_economics_2026   -> the_subject_business_economics_2026.csv (28 filas)
✔ latam_2026                -> the_latam_ranking_2026.csv (28 filas)


## 5. Próximos pasos

* Comparar los pilares (`teaching`, `research_env`, `research_quality`, `industry`, `international`) entre los tres rankings.
* Cruzar con datos internos de la Universidad del Pacífico para análisis competitivo.
* Agendar una ejecución periódica (p.ej. mensual) para detectar cambios.

Si THE vuelve a cambiar la estructura del sitio, el único punto que normalmente necesita ajustarse es el 
diccionario **`COLUMNAS_CSS`** y la lógica dentro de `scrapear_ranking`.